In [ ]:
# To install the google-genai
%pip install google-genai

In [ ]:
# To import all the libraries and fetch api_key from .env file
import os
import dotenv
from google import genai

# Here the URL is connecting to the Google Server via OpenAI endpoint
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"

dotenv.load_dotenv() # this line reads environment variables from .env file and loads them into the process environment (virtual environment) 
gemini_api_key = os.getenv('GEMINI_API_KEY') # to get the environment variable (GEMINI_API_KEY) from the process environment to which the environment variables are already loaded

if not gemini_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not gemini_api_key.startswith("AQ."):
    print("An API key was found, but it doesn't start AQ.")
else:
    print("API key found and looks good so far!")



API key found and looks good so far!


## Approach 1

1. Using Raw HTTP Request

2. GEMINI_BASE_URL contains /openai/chat/completions, and payload passes text inside a messages array structured with {"role": "user", "content": "..."}. This is exactly how the OpenAI API functions. Google built this translation proxy endpoint so developers who already wrote applications for GPT-4 could swap out the backend URL to Gemini without rewriting their code.


In [60]:
import requests

headers = {"Authorization": f"Bearer {gemini_api_key}", "Content-Type": "application/json"}

payload = {
    "model": "gemini-3.1-flash-lite",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}]
}

print(payload)

response = requests.post(
    GEMINI_BASE_URL,
    headers=headers,
    json=payload
)

response.json()


{'model': 'gemini-3.1-flash-lite', 'messages': [{'role': 'user', 'content': 'Tell me a fun fact'}]}


{'choices': [{'finish_reason': 'stop',
   'index': 0,
   'message': {'content': 'Sea otters hold hands when they sleep so they don’t drift away from each other. They sometimes even form "rafts" by linking together in large groups!',
    'extra_content': {'google': {'thought_signature': 'EjQKMgEMOdbHAKwDOCGBQjGhOOF69RZe9UGozyBRMPNnGRZpazcZUrXcCcXtvmhWqiPYwvQl'}},
    'role': 'assistant'}}],
 'created': 1782895106,
 'id': '_9FEaqLAGpy2juMPnv_62Ak',
 'model': 'gemini-3.1-flash-lite',
 'object': 'chat.completion',
 'usage': {'completion_tokens': 33, 'prompt_tokens': 6, 'total_tokens': 39}}

In [61]:
response.json()['choices'][0]['message']['content']

'Sea otters hold hands when they sleep so they don’t drift away from each other. They sometimes even form "rafts" by linking together in large groups!'

## Approach 2

1. Using Google SDK

2. The SDK automatically handles low-level details like formatting the HTTP payload, managing timeouts, handling network retries, and securely appending 'AQ.' api key behind the scenes

In [35]:
client = genai.Client(api_key=gemini_api_key)

response = client.models.generate_content(
    model='gemini-3.1-flash-lite',
    contents='Hello, Tell me a fun fact about earth'
)

print(response)


sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text="""Here is a fun fact for you: **Earth is not a perfect sphere!**

Because of the force created by its constant rotation, the Earth bulges at the center. This makes it an **"oblate spheroid"** rather than a perfect ball. 

If you were to measure the Earth from pole to pole, it would be about **43 kilometers (27 miles) shorter** than if you measured it around the equator! Essentially, the Earth has a little bit of a "spare tire" around its middle.""",
        thought_signature=b'\x124\n2\x01\x0c9\xd6\xc7\xd2e\xe5\x88\x95\x90\x97\xc0\xec\xcf\x9e\xc0\xf9\xce\xa7\x93\x8dT\x00\x80\x8fDkbU\x8cBV\x86J\xddJR\x8f\x00!\xbbu\x01\x1f\xd1V\xf2\xde\xe3'
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-3.1-flash-lite' prompt_feedback=None response_id='scdEat7QPIvEjuMPst2ZyQ4' usage

In [36]:
print(response.text)

Here is a fun fact for you: **Earth is not a perfect sphere!**

Because of the force created by its constant rotation, the Earth bulges at the center. This makes it an **"oblate spheroid"** rather than a perfect ball. 

If you were to measure the Earth from pole to pole, it would be about **43 kilometers (27 miles) shorter** than if you measured it around the equator! Essentially, the Earth has a little bit of a "spare tire" around its middle.
